In [1]:
import requests
from bs4 import BeautifulSoup
import warnings

# 忽略 SSL 警告
warnings.filterwarnings("ignore")

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
}

# 設定測試目標
TARGETS = [
    {
        "name": "TVBS 新聞網首頁 (L3)",
        "url": "https://news.tvbs.com.tw/",
        "selectors": [
            "div.hero_main a.swiper-slide",  # 假設 1: 輪播大圖
            "div.index_big a",               # 假設 2: 舊版大圖結構
            "div.focus_news a",              # 假設 3: 焦點新聞區
            "section.index_focus a"          # 假設 4: 新版焦點區
        ]
    },
    {
        "name": "TVBS 即時新聞 (L1)",
        "url": "https://news.tvbs.com.tw/realtime",
        "selectors": [
            ".news_list .list li a",         # 假設 1: 標準列表
            "div.realtime_list ul li a",     # 假設 2: 區塊列表
            "div.list ul li a",              # 假設 3: 通用列表
            "a.realtime_news_list"           # 假設 4: 直接 class
        ]
    }
]

print("🚀 開始 TVBS 結構診斷...")

for target in TARGETS:
    print(f"\n📡 測試：{target['name']}")
    print(f"   網址: {target['url']}")
    
    try:
        res = requests.get(target['url'], headers=HEADERS, timeout=15, verify=False)
        if res.status_code != 200:
            print(f"   🔥 連線失敗: {res.status_code}")
            continue
            
        soup = BeautifulSoup(res.text, 'lxml')
        
        # 測試每個選擇器
        success = False
        for sel in target['selectors']:
            items = soup.select(sel)
            if len(items) > 0:
                # 嘗試抓文字，過濾掉空連結
                valid_items = [i for i in items if i.get_text(strip=True)]
                if valid_items:
                    print(f"   ✅ 命中選擇器: '{sel}'")
                    print(f"      抓到筆數: {len(valid_items)}")
                    print(f"      範例標題: {valid_items[0].get_text(strip=True)[:20]}...")
                    success = True
                    # 找到一個有效的就換下一個目標，避免洗版
                    # break 
                else:
                     print(f"   ⚠️ 選擇器 '{sel}' 抓到元素但無文字 (可能是圖片連結)")
            else:
                print(f"   ❌ 選擇器無效: '{sel}'")
        
        if not success:
            print("   😭 所有預設選擇器都失敗！")

    except Exception as e:
        print(f"   💥 發生錯誤: {e}")

print("\n診斷結束。請將結果貼給我。")

🚀 開始 TVBS 結構診斷...

📡 測試：TVBS 新聞網首頁 (L3)
   網址: https://news.tvbs.com.tw/
   ❌ 選擇器無效: 'div.hero_main a.swiper-slide'
   ❌ 選擇器無效: 'div.index_big a'
   ❌ 選擇器無效: 'div.focus_news a'
   ❌ 選擇器無效: 'section.index_focus a'
   😭 所有預設選擇器都失敗！

📡 測試：TVBS 即時新聞 (L1)
   網址: https://news.tvbs.com.tw/realtime
   ✅ 命中選擇器: '.news_list .list li a'
      抓到筆數: 782
      範例標題: 獨／懷胎15個月才出生！「最帥乩童」黃新...
   ❌ 選擇器無效: 'div.realtime_list ul li a'
   ✅ 命中選擇器: 'div.list ul li a'
      抓到筆數: 843
      範例標題: 即時...
   ❌ 選擇器無效: 'a.realtime_news_list'

診斷結束。請將結果貼給我。
